# IN-DEPTH END-TO-END SUPERSTORE DATA ENGINEERING & ANALYTICS WORKBOOK
**Database Dialect Core:** SQLite Relational Architecture (Fully Portable)

### Project Overview & Structural Methodology
This document is configured to provide an advanced, step-by-step technical pipeline that replicates real-world corporate engineering practices. We migrate an unorganized, flat schema database layer (`superstore_raw`) into an optimized operational ecosystem. This transformation enforces relational dependencies, data type safety, and entity index normalization before running advanced subqueries, nested CTE loops, and analytic window calculations.

## STAGE 1: RUNTIME CONFIGURATION & STAGING ENVIRONMENT INGESTION
First, initialize the local in-memory relational workspace connection and simulate a complete flat-file table staging infrastructure (`superstore_raw`) using comprehensive enterprise records.

In [13]:
import sqlite3
import pandas as pd

# Establish the local portable operational connection layer
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Construct a highly descriptive, structural sample staging database block
enterprise_staging_data = {
    'Row ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Order ID': ['CA-2024-152156', 'CA-2024-152156', 'CA-2024-138688', 'US-2024-108966', 'US-2024-108966', 'CA-2024-115812', 'CA-2024-115812', 'CA-2024-115812', 'CA-2024-167164', 'CA-2024-143336'],
    'Order Date': ['11/08/2024', '11/08/2024', '06/12/2024', '10/11/2024', '10/11/2024', '06/09/2024', '06/09/2024', '06/09/2024', '05/12/2024', '08/27/2024'],
    'Ship Date': ['11/11/2024', '11/11/2024', '06/16/2024', '10/18/2024', '10/18/2024', '06/14/2024', '06/14/2024', '06/14/2024', '05/15/2024', '09/01/2024'],
    'Ship Mode': ['Second Class', 'Second Class', 'Second Class', 'Standard Class', 'Standard Class', 'Standard Class', 'Standard Class', 'Standard Class', 'Second Class', 'Second Class'],
    'Customer ID': ['CG-12520', 'CG-12520', 'DV-13045', 'SO-20335', 'SO-20335', 'BH-11710', 'BH-11710', 'BH-11710', 'AG-10270', 'ZD-21925'],
    'Customer Name': ['Claire Gute', 'Claire Gute', 'Darrin Van Huff', "Sean O'Donnell", "Sean O'Donnell", 'Brosina Hoffman', 'Brosina Hoffman', 'Brosina Hoffman', 'Alejandro Grove', 'Zuschlich Donelson'],
    'Segment': ['Consumer', 'Consumer', 'Corporate', 'Consumer', 'Consumer', 'Consumer', 'Consumer', 'Consumer', 'Consumer', 'Consumer'],
    'City': ['Henderson', 'Henderson', 'Los Angeles', 'Fort Lauderdale', 'Fort Lauderdale', 'Los Angeles', 'Los Angeles', 'Los Angeles', 'West Jordan', 'San Francisco'],
    'State': ['Kentucky', 'Kentucky', 'California', 'Florida', 'Florida', 'California', 'California', 'California', 'Utah', 'California'],
    'Product ID': ['FUR-BO-10001798', 'FUR-CH-10000454', 'OFF-LA-10000240', 'FUR-TA-10000577', 'OFF-ST-10000760', 'FUR-FU-10001487', 'OFF-AR-10002833', 'TEC-PH-10002275', 'OFF-BI-10003910', 'OFF-AP-10002892'],
    'Category': ['Furniture', 'Furniture', 'Office Supplies', 'Furniture', 'Office Supplies', 'Furniture', 'Office Supplies', 'Technology', 'Office Supplies', 'Office Supplies'],
    'Sub-Category': ['Bookcases', 'Chairs', 'Labels', 'Tables', 'Storage', 'Furnishings', 'Art', 'Phones', 'Binders', 'Appliances'],
    'Product Name': ['Bush Somerset Bookcase', 'Hon Deluxe Fabric Upholstered Chairs', 'Self-Adhesive Address Labels', 'Bretford Conference Table', "Eldon Fold 'N Roll Cart", 'Floodlight Pixel Lamp', 'Sanford Canvas Markers', 'Mitel 5320 IP Phone', 'DXL Angle-View Binders', 'Hoover Portable Vacuum'],
    'Sales': [261.96, 731.94, 14.62, 957.57, 22.36, 48.86, 7.28, 907.15, 18.50, 114.90],
    'Quantity': [2, 3, 2, 5, 2, 7, 4, 6, 3, 5],
    'Discount': [0.00, 0.00, 0.00, 0.45, 0.20, 0.00, 0.00, 0.20, 0.00, 0.00],
    'Profit': [41.91, 219.58, 6.87, -383.03, 2.51, 14.17, 1.97, 90.72, 9.25, 34.47]
}

df_staging = pd.DataFrame(enterprise_staging_data)
df_staging.to_sql('superstore_raw', conn, if_exists='replace', index=False)
print('[SUCCESS] Raw staging database loaded.')

[SUCCESS] Raw staging database loaded.


## STAGE 2: STEP-BY-STEP MULTI-TABLE MIGRATION & SCHEMATIC NORMALIZATION
In this step, we decouple the flat table into 3 separate entities to ensure structural isolation. We enforce explicit datatypes, clear structural constraints, and clean target records via `SELECT DISTINCT` routines.

In [14]:
# Step A: DDL Generation - Building Physical Normalized Blueprints
cursor.execute('''
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL,
    segment TEXT,
    city TEXT,
    state TEXT
);''')

cursor.execute('''
CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT,
    sub_category TEXT
);''')

cursor.execute('''
CREATE TABLE orders (
    row_id INTEGER PRIMARY KEY,
    order_id TEXT NOT NULL,
    order_date TEXT NOT NULL,
    ship_date TEXT,
    ship_mode TEXT,
    customer_id TEXT,
    product_id TEXT,
    sales REAL CHECK(sales >= 0),
    quantity INTEGER CHECK(quantity > 0),
    discount REAL CHECK(discount >= 0 AND discount <= 1),
    profit REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);''')

print('[SUCCESS] Relational table blueprints generated.')

[SUCCESS] Relational table blueprints generated.


In [15]:
# Step B: DML Extraction Pipeline - Loading Unique Attributes sequentially
cursor.execute('''
INSERT INTO customers (customer_id, customer_name, segment, city, state)
SELECT DISTINCT `Customer ID`, `Customer Name`, `Segment`, `City`, `State` 
FROM superstore_raw;''')

cursor.execute('''
INSERT INTO products (product_id, product_name, category, sub_category)
SELECT DISTINCT `Product ID`, `Product Name`, `Category`, `Sub-Category` 
FROM superstore_raw;''')

cursor.execute('''
INSERT INTO orders (row_id, order_id, order_date, ship_date, ship_mode, customer_id, product_id, sales, quantity, discount, profit)
SELECT `Row ID`, `Order ID`, `Order Date`, `Ship Date`, `Ship Mode`, `Customer ID`, `Product ID`, `Sales`, `Quantity`, `Discount`, `Profit` 
FROM superstore_raw;''')

conn.commit()
print('[SUCCESS] Distinct multi-table migration completed successfully.')

[SUCCESS] Distinct multi-table migration completed successfully.


### VERIFYING MIGRATED TABLE LAYOUTS
Review the distinct outputs below to verify the database normalization results.

In [16]:
print('--- CLEANED CUSTOMERS REGISTER TABLE ---')
display(pd.read_sql_query('SELECT * FROM customers;', conn))

print('\n--- CLEANED PRODUCTS REGISTRY TABLE ---')
display(pd.read_sql_query('SELECT * FROM products LIMIT 5;', conn))

print('\n--- TRANSGRESSED ORDERS TRANSACTION TABLE ---')
display(pd.read_sql_query('SELECT row_id, order_id, customer_id, product_id, sales, profit FROM orders LIMIT 5;', conn))

--- CLEANED CUSTOMERS REGISTER TABLE ---


,customer_id,customer_name,segment,city,state
0,CG-12520,Claire Gute,Consumer,Henderson,Kentucky
1,DV-13045,Darrin Van Huff,Corporate,Los Angeles,California
2,SO-20335,Sean O'Donnell,Consumer,Fort Lauderdale,Florida
3,BH-11710,Brosina Hoffman,Consumer,Los Angeles,California
4,AG-10270,Alejandro Grove,Consumer,West Jordan,Utah
5,ZD-21925,Zuschlich Donelson,Consumer,San Francisco,California



--- CLEANED PRODUCTS REGISTRY TABLE ---


,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,Hon Deluxe Fabric Upholstered Chairs,Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels,Office Supplies,Labels
3,FUR-TA-10000577,Bretford Conference Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart,Office Supplies,Storage



--- TRANSGRESSED ORDERS TRANSACTION TABLE ---


,row_id,order_id,customer_id,product_id,sales,profit
0,1,CA-2024-152156,CG-12520,FUR-BO-10001798,261.96,41.91
1,2,CA-2024-152156,CG-12520,FUR-CH-10000454,731.94,219.58
2,3,CA-2024-138688,DV-13045,OFF-LA-10000240,14.62,6.87
3,4,US-2024-108966,SO-20335,FUR-TA-10000577,957.57,-383.03
4,5,US-2024-108966,SO-20335,OFF-ST-10000760,22.36,2.51


## STAGE 3: INTERMEDIATE TO ADVANCED ANALYTICAL RUNTIME TASKS
Executing strategic business calculations using deep Subqueries, atomic Common Table Expressions, and scalable Window Functions.

In [17]:
print('--- ADVANCED TASK 1: Orders Exceeding Global Mean Sales (Subquery) ---')
at1_sql = '''
SELECT order_id, customer_id, sales 
FROM orders 
WHERE sales > (SELECT AVG(sales) FROM orders);
'''
display(pd.read_sql_query(at1_sql, conn))

print('\n--- ADVANCED TASK 2: Peak Transaction Absolute Single Line Value Per Customer Account (Subquery) ---')
at2_sql = '''
SELECT o.customer_id, o.order_id, o.sales
FROM orders o
WHERE o.sales = (
    SELECT MAX(sub.sales) 
    FROM orders sub 
    WHERE sub.customer_id = o.customer_id
);
'''
display(pd.read_sql_query(at2_sql, conn))

--- ADVANCED TASK 1: Orders Exceeding Global Mean Sales (Subquery) ---


,order_id,customer_id,sales
0,CA-2024-152156,CG-12520,731.94
1,US-2024-108966,SO-20335,957.57
2,CA-2024-115812,BH-11710,907.15



--- ADVANCED TASK 2: Peak Transaction Absolute Single Line Value Per Customer Account (Subquery) ---


,customer_id,order_id,sales
0,CG-12520,CA-2024-152156,731.94
1,DV-13045,CA-2024-138688,14.62
2,SO-20335,US-2024-108966,957.57
3,BH-11710,CA-2024-115812,907.15
4,AG-10270,CA-2024-167164,18.50
5,ZD-21925,CA-2024-143336,114.90


In [18]:
print('--- ADVANCED TASK 3 & 4: Total Revenue Tracking vs Customer Cohort Mean (CTE + Subquery) ---')
at3_sql = '''
WITH CustomerSalesSummary AS (
    SELECT customer_id, SUM(sales) AS total_customer_revenue
    FROM orders
    GROUP BY customer_id
)
SELECT css.customer_id, c.customer_name, css.total_customer_revenue
FROM CustomerSalesSummary css
JOIN customers c ON css.customer_id = c.customer_id
WHERE css.total_customer_revenue > (SELECT AVG(total_customer_revenue) FROM CustomerSalesSummary);
'''
display(pd.read_sql_query(at3_sql, conn))

print('\n--- ADVANCED TASK 5 & 6: Sequential Lifecycle Sequence & Cohort Rankings (Window Functions) ---')
at5_sql = '''
SELECT 
    customer_id,
    order_id,
    sales,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS internal_order_sequence,
    DENSE_RANK() OVER (ORDER BY sales DESC) AS global_revenue_line_rank
FROM orders;
'''
display(pd.read_sql_query(at5_sql, conn))

--- ADVANCED TASK 3 & 4: Total Revenue Tracking vs Customer Cohort Mean (CTE + Subquery) ---


,customer_id,customer_name,total_customer_revenue
0,BH-11710,Brosina Hoffman,963.29
1,CG-12520,Claire Gute,993.90
2,SO-20335,Sean O'Donnell,979.93



--- ADVANCED TASK 5 & 6: Sequential Lifecycle Sequence & Cohort Rankings (Window Functions) ---


,customer_id,order_id,sales,internal_order_sequence,global_revenue_line_rank
0,AG-10270,CA-2024-167164,18.50,1,8
1,BH-11710,CA-2024-115812,907.15,1,2
2,BH-11710,CA-2024-115812,48.86,2,6
3,BH-11710,CA-2024-115812,7.28,3,10
4,CG-12520,CA-2024-152156,731.94,1,3
5,CG-12520,CA-2024-152156,261.96,2,4
6,DV-13045,CA-2024-138688,14.62,1,9
7,SO-20335,US-2024-108966,957.57,1,1
8,SO-20335,US-2024-108966,22.36,2,7
9,ZD-21925,CA-2024-143336,114.90,1,5


## STAGE 4: STEP 3 - FINAL COMBINED COMPREHENSIVE BUSINESS REPORT
This query joins our structured tables, aggregates revenue via an inline CTE, and applies an analytical window function to generate final customer ranks.

In [19]:
master_analytics_ledger_sql = '''
WITH GrossSalesCTE AS (
    SELECT customer_id, SUM(sales) AS lifetime_gross_sales
    FROM orders
    GROUP BY customer_id
),
RankedPerformanceCTE AS (
    SELECT 
        customer_id,
        lifetime_gross_sales,
        DENSE_RANK() OVER (ORDER BY lifetime_gross_sales DESC) AS strategic_customer_rank
    FROM GrossSalesCTE
)
SELECT 
    c.customer_name,
    rp.lifetime_gross_sales,
    rp.strategic_customer_rank
FROM RankedPerformanceCTE rp
JOIN customers c ON rp.customer_id = c.customer_id
ORDER BY rp.strategic_customer_rank ASC;
'''
print('--- FINAL COMBINED OPERATIONAL REPORT TABLE ---')
display(pd.read_sql_query(master_analytics_ledger_sql, conn))

--- FINAL COMBINED OPERATIONAL REPORT TABLE ---


,customer_name,lifetime_gross_sales,strategic_customer_rank
0,Claire Gute,993.90,1
1,Sean O'Donnell,979.93,2
2,Brosina Hoffman,963.29,3
3,Zuschlich Donelson,114.90,4
4,Alejandro Grove,18.50,5
5,Darrin Van Huff,14.62,6


## STAGE 5: MINI PROJECT COMPLETION - TARGETED CUSTOMER INSIGHTS
Executing the individual business case queries required to complete the case study assignment module.

In [20]:
print('--- MINI-PROJECT TASK 1: TOP 5 REVENUE GENERATING CUSTOMERS ---')
mp_t1 = '''
WITH Agg AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id)
SELECT a.customer_id, c.customer_name, a.ts as total_sales, DENSE_RANK() OVER(ORDER BY a.ts DESC) as rk
FROM Agg a JOIN customers c ON a.customer_id = c.customer_id LIMIT 5;'''
display(pd.read_sql_query(mp_t1, conn))

print('\n--- MINI-PROJECT TASK 2: BOTTOM 5 REVENUE GENERATING CUSTOMERS ---')
mp_t2 = '''
WITH Agg AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id)
SELECT a.customer_id, c.customer_name, a.ts as total_sales, DENSE_RANK() OVER(ORDER BY a.ts ASC) as rk
FROM Agg a JOIN customers c ON a.customer_id = c.customer_id LIMIT 5;'''
display(pd.read_sql_query(mp_t2, conn))

print('\n--- MINI-PROJECT TASK 3: CHURN-RISK CUSTOMERS WITH ONLY ONE SINGLE TRANSACTION ---')
mp_t3 = '''
SELECT o.customer_id, c.customer_name, COUNT(DISTINCT o.order_id) AS total_lifetime_orders
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1;'''
display(pd.read_sql_query(mp_t3, conn))

print('\n--- MINI-PROJECT TASK 4: ACCOUNT BASE PERFORMING ABOVE THE GENERAL COHORT MEAN ---')
mp_t4 = '''
WITH AggregateSales AS (SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id)
SELECT a.customer_id, c.customer_name, a.total_sales
FROM AggregateSales a JOIN customers c ON a.customer_id = c.customer_id
WHERE a.total_sales > (SELECT AVG(total_sales) FROM AggregateSales);'''
display(pd.read_sql_query(mp_t4, conn))

print('\n--- MINI-PROJECT TASK 5: PEAK SINGLE CART ORDER VALUE ACHIEVED PER INDIVIDUAL CUSTOMER ACCOUNT ---')
mp_t5 = '''
WITH SingleCartTotals AS (
    SELECT customer_id, order_id, SUM(sales) AS single_order_total_value
    FROM orders GROUP BY customer_id, order_id
)
SELECT sct.customer_id, c.customer_name, MAX(sct.single_order_total_value) AS peak_historical_order_value
FROM SingleCartTotals sct
JOIN customers c ON sct.customer_id = c.customer_id
GROUP BY sct.customer_id, c.customer_name;'''
display(pd.read_sql_query(mp_t5, conn))

--- MINI-PROJECT TASK 1: TOP 5 REVENUE GENERATING CUSTOMERS ---


,customer_id,customer_name,total_sales,rk
0,CG-12520,Claire Gute,993.90,1
1,SO-20335,Sean O'Donnell,979.93,2
2,BH-11710,Brosina Hoffman,963.29,3
3,ZD-21925,Zuschlich Donelson,114.90,4
4,AG-10270,Alejandro Grove,18.50,5



--- MINI-PROJECT TASK 2: BOTTOM 5 REVENUE GENERATING CUSTOMERS ---


,customer_id,customer_name,total_sales,rk
0,DV-13045,Darrin Van Huff,14.62,1
1,AG-10270,Alejandro Grove,18.50,2
2,ZD-21925,Zuschlich Donelson,114.90,3
3,BH-11710,Brosina Hoffman,963.29,4
4,SO-20335,Sean O'Donnell,979.93,5



--- MINI-PROJECT TASK 3: CHURN-RISK CUSTOMERS WITH ONLY ONE SINGLE TRANSACTION ---


,customer_id,customer_name,total_lifetime_orders
0,AG-10270,Alejandro Grove,1
1,BH-11710,Brosina Hoffman,1
2,CG-12520,Claire Gute,1
3,DV-13045,Darrin Van Huff,1
4,SO-20335,Sean O'Donnell,1
5,ZD-21925,Zuschlich Donelson,1



--- MINI-PROJECT TASK 4: ACCOUNT BASE PERFORMING ABOVE THE GENERAL COHORT MEAN ---


,customer_id,customer_name,total_sales
0,BH-11710,Brosina Hoffman,963.29
1,CG-12520,Claire Gute,993.90
2,SO-20335,Sean O'Donnell,979.93



--- MINI-PROJECT TASK 5: PEAK SINGLE CART ORDER VALUE ACHIEVED PER INDIVIDUAL CUSTOMER ACCOUNT ---


,customer_id,customer_name,peak_historical_order_value
0,AG-10270,Alejandro Grove,18.50
1,BH-11710,Brosina Hoffman,963.29
2,CG-12520,Claire Gute,993.90
3,DV-13045,Darrin Van Huff,14.62
4,SO-20335,Sean O'Donnell,979.93
5,ZD-21925,Zuschlich Donelson,114.90
